# 🤖 Notebook 04: XGBoost 负荷预测

## 学习目标
1. 理解梯度提升 (Gradient Boosting) 的核心思想
2. 学会用 XGBoost 做时序预测
3. 理解 TimeSeriesSplit 和 look-ahead bias
4. 掌握模型评估指标 (MAE/RMSE/MAPE/R²)
5. 读懂特征重要性

## 梯度提升 (Gradient Boosting) 原理

### 类比: 一群专家协作

想象你要预测明天的用电量。你找来了 100 个专家:

**专家 1**: "全年均值是 50000MW，我预测 50000MW"
→ 残差 (错误) = 实际值 - 50000

**专家 2**: "让我专门修正专家 1 的错误。夏天加 10000，冬天加 5000"
→ 把专家 1 的错误又减少了一些

**专家 3**: "我再修正前两位的错误。周末减 8000，工作日下午加 5000"
→ 错误继续减少

...重复 100 轮...

**最终预测 = 专家 1 + 专家 2 + 专家 3 + ... + 专家 100**

核心思想: 每一轮都在学习前一轮的**残差 (错误)**，
逐步逼近真实值。

### 对比持续法

| 方法 | 原理 | 优点 | 缺点 |
|------|------|------|------|
| **持续法** | 昨天=今天 | 零训练, 零参数 | 不会学习任何模式 |
| **XGBoost** | 100棵树逐步修正 | 自动学习复杂模式 | 需要训练, 可能过拟合 |

如果 XGBoost 的 MAE > 持续法的 MAE，说明 XGBoost 配置有问题。

In [ ]:
import sys
sys.path.insert(0, '..')
from pipeline.data_loader import create_loader
from pipeline.cleaner import clean_data
from pipeline.features import FeatureEngineer
from pipeline.forecaster import XGBoostForecaster
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook'

In [ ]:
# ── 数据准备 ──────────────────────────────────
loader = create_loader('owid')
df = loader.load_data()
df = clean_data(df)

# 特征工程 — Tier 1 (先跑通核心特征)
engineer = FeatureEngineer()
df = engineer.add_tier1_features(df)

feature_cols = engineer.get_feature_columns('tier1')
X = df[feature_cols]
y = df['load_mw']

print(f"训练数据: {len(X)} 行 × {len(feature_cols)} 特征")
print(f"特征: {feature_cols}")
print(f"目标: load_mw (日均负荷, MW)")

---
## 步骤 1: 训练 XGBoost 模型

`train_evaluate()` 内部自动使用 TimeSeriesSplit 切割数据，
确保训练永远在测试之前——防止 look-ahead bias。

In [ ]:
forecaster = XGBoostForecaster(
    n_estimators=100,   # 100 棵树（迭代次数）
    max_depth=6,        # 每棵树最大深度
    learning_rate=0.1,  # 学习率
)

result = forecaster.train_evaluate(X, y, n_splits=5, gap=0)

metrics = result['metrics']
print(f"\n═══ XGBoost 评估结果 ═══")
print(f"MAE:   {metrics['mae']:,.0f} MW")
print(f"RMSE:  {metrics['rmse']:,.0f} MW")
print(f"MAPE:  {metrics['mape']:.1f}%")
print(f"R²:    {metrics['r2']:.3f}")

### 📖 评估指标解读

**MAE (平均绝对误差):**
$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$
所有预测偏差的绝对值的均值。
单位与原始数据相同（MW）。
→ "平均来说，预测偏差 ± X MW"

**RMSE (均方根误差):**
$$RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$
先平方 → 对大误差的惩罚更重 → 再开方保持量纲。
RMSE ≥ MAE，两者差距大说明存在少量极大误差。

**MAPE (平均绝对百分比误差):**
$$MAPE = \frac{100}{n}\sum_{i=1}^{n} \left|\frac{y_i - \hat{y}_i}{y_i}\right|$$
方便跨数据集比较（不同市场、不同单位），
MAPE=5% 表示预测值平均偏离真实值 5%。

**R² (决定系数):**
$$R² = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
衡量模型比均值好多少。
R²=1 表示完美预测; R²=0 表示不均值好; R²<0 比瞎猜还差。

In [ ]:
# ── 对比持续法 ────────────────────────────────
from pipeline.forecaster import persistence_forecast
persist = persistence_forecast(df)
persist_mae = (persist - df['load_mw']).abs().mean()

print(f"═══ 模型对比 ═══")
print(f"持续法 MAE:   {persist_mae:,.0f} MW")
print(f"XGBoost MAE:  {metrics['mae']:,.0f} MW")

improvement = (persist_mae - metrics['mae']) / persist_mae * 100
if improvement > 0:
    print(f"XGBoost 提升了 {improvement:.1f}% — 比持续法更好!")
else:
    print(f"⚠ XGBoost 没比持续法好，可能特征工程或参数有问题")

---
## 步骤 2: 特征重要性

XGBoost 内置了特征重要性 (Feature Importance)，
告诉你哪些特征对预测贡献最大。

使用的是 **gain** 指标:
特征在所有树的所有分割点上的平均信息增益。

In [ ]:
importance = result['feature_importance']
import_df = pd.DataFrame(
    sorted(importance.items(), key=lambda x: x[1], reverse=True),
    columns=['特征', '重要性 (gain)']
)

print("特征重要性排名 (gain):")
for i, row in import_df.iterrows():
    bar = '█' * int(row['重要性 (gain)'] / import_df['重要性 (gain)'].max() * 20)
    print(f"  {row['特征']:20s} {bar} ({row['重要性 (gain)']:.1f})")

print(f"\n📖 解读: {import_df.iloc[0]['特征']} 是最重要的特征。")
print("这通常意味着负荷有很强的时间相关性。")

---
## 步骤 3: 预测 vs 实际 可视化

In [ ]:
# 用训练好的模型做全量预测
all_pred = forecaster.predict(X)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('实际 vs XGBoost 预测', '预测偏差分布'),
                    row_heights=[0.6, 0.4])

# 子图1: 实际 vs 预测
fig.add_trace(go.Scatter(x=df['timestamp'], y=df['load_mw'], name='实际', mode='lines+markers',
                         line=dict(color='#1f77b4', width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=df['timestamp'], y=all_pred, name='XGBoost', mode='lines+markers',
                         line=dict(color='#ff7f0e', width=1.5, dash='dash')), row=1, col=1)

# 子图2: 偏差分布
errors = all_pred - df['load_mw'].values
fig.add_trace(go.Histogram(x=errors, name='预测偏差', nbinsx=20,
                           marker_color='#2ca02c'), row=2, col=1)
fig.add_vline(x=0, line_dash='dash', line_color='red', row=2, col=1)

fig.update_layout(title='XGBoost 负荷预测', hovermode='x unified', height=600)
fig.update_yaxes(title_text='负荷 (MW)', row=1, col=1)
fig.update_yaxes(title_text='频次', row=2, col=1)
fig.show()

## 📝 学习笔记

- XGBoost 通过 100 棵树逐步修正错误来逼近真实值
- TimeSeriesSplit 防止 look-ahead bias — 永远用过去预测未来
- 特征重要性告诉你模型的"思考过程"
- 永远和持续法对比 — 如果 XGBoost 不如持续法，回去检查

## 🤔 反思题

1. 如果 MAPE > 10%，可能是什么原因？
2. 哪三个超参数对 XGBoost 性能影响最大？
3. 年级数据和小时级数据，哪个更适合 XGBoost 学习？为什么？

### 思考题

1. TimeSeriesSplit 的 gap=24 是防什么的？如果 gap=0，会发生什么？
2. MAE 告诉我们预测偏差多少 MW。如果两个模型 MAE 相同，怎么判断哪个更好？
3. 特征重要性排名第一的特征是什么？这符合你对电力系统的直觉吗？

